In [ ]:
# Import Dependencies
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [ ]:
# Define the eBay filters dictionary
ebay_filters = {
    "item_conditions": {
        "New": 1000,
        "Open box": 1500,
        "Used": 3000,
        "Very Good Plus (VG+)": 4000,
        "Very Good (VG)": 5000,
        "Good Plus (G+)": 6000,
        "Good (G)": 7000,
        "For Parts or Not Working": 7000
    },
    "item_locations": {
        "Domestic": 1,
        "International": 2,
        "Continent": 3,
    },
    "directories": {
        "No Directory": 0,
        "Music": 11233,
        "Entertainment Memorabilia": 45,
        "Collectibles": 1,
    },
    "categories": {
        "No Category": 0,
        "Records": 176985,         # Music > Records (LP, 12", 7" vinyl)
        "45 RPM (7\")": 309,        # 7-inch singles
        "12\" Singles": 13993,      # 12-inch singles / maxi singles
        "LPs (Albums)": 33,         # Full-length albums
        "78 RPM": 261,              # 78 RPM shellac / early vinyl
        "Box Sets": 80107,          # Vinyl box sets
        "Picture Discs": 41588,     # Limited / picture discs
        "Test Pressings": 41589,    # Test pressings
    },
    "sort_order": {
        "Best Match": 12,
        "Time: ending soonest": 1,
        "Time: newly listed": 10,
        "Price + Shipping: lowest first": 15,
        "Price + Shipping: highest first": 16,
        "Distance: nearest first": 7
    }
}


In [ ]:
# --- Customise your search here ---
search_term = 'pink floyd dark side of the moon vinyl'
min_price   = '5'    # Minimum price in your local currency
max_price   = '50'   # Maximum price — adjust for bargain threshold

# Define the base URL for the eBay search
url = 'https://www.ebay.com/sch/i.html'

# Define the query parameters for the search request
params = {
    '_from': 'R40',
    '_nkw': search_term,
    'LH_ItemCondition': ebay_filters['item_conditions']['Used'],  # Used condition
    'LH_PrefLoc': ebay_filters['item_locations']['International'],
    '_udlo': min_price,    # Minimum price
    '_udhi': max_price,    # Maximum price
    '_dcat': ebay_filters['directories']['Music'],                # Music directory
    '_sacat': ebay_filters['categories']['Records'],              # Records category (LP / vinyl)
    '_sop': ebay_filters['sort_order']['Time: newly listed'],     # Sort: newly listed first
    'LH_Sold': '1',    # Only sold / completed listings
    'LH_Complete': '1',
    'LH_BIN': '1',     # Include Buy It Now listings
    'LH_Auction': '1', # Include Auction listings
    'LH_BO': '0',      # Exclude "Best Offer only" listings
    'LH_FS': '0',      # Free shipping optional — set to '1' to require it
    '_ipg': '240',     # Items per page (max 240)
    'rt': 'nc'         # No-cache: ensure fresh results
}


In [ ]:
# Create a prepared request to inspect the full URL before scraping
request = requests.Request('GET', url, params=params)
prepared_request = request.prepare()
print(prepared_request.url)


In [ ]:
# Initialize variables
page_number = 0
items_list = []

# Loop over pages
while True:

    page_number += 1
    print(f'Scraping page: {page_number}')

    params['pgn'] = page_number

    # Send GET request to eBay with the defined parameters
    response = requests.get(url, params=params)
    html_content = response.text

    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the pagination next button
    next_button = soup.find('button', class_='pagination__next', type='next')

    # Check whether there are more pages
    if next_button and next_button.get('aria-disabled') == 'true':
        print('No more pages to scrape')
        break
    else:
        # Extract item wrappers
        items = soup.find_all('div', class_='s-item__wrapper clearfix')

        # Skip the first two placeholder items injected by eBay
        for item in items[2:]:
            title     = item.find('div', class_='s-item__title').text
            price     = item.find('span', class_='s-item__price').text
            link      = item.find('a', class_='s-item__link')['href'].split('?')[0]
            image_url = (
                item
                .find('div', class_='s-item__image-wrapper image-treatment')
                .find('img')
                .get('src', 'No image URL')
            )

            item_dict = {
                'Title':      title,
                'Price':      price,
                'Link':       link,
                'Image Link': image_url
            }

            items_list.append(item_dict)


In [ ]:
print(f'Total raw listings scraped: {len(items_list)}')


In [ ]:
# Turn the list into a Pandas DataFrame
items_df = pd.DataFrame(items_list)
items_df


In [ ]:
# Define forbidden terms — listings containing any of these are excluded
forbidden_terms = [
    # Physical condition issues
    'warped',
    'scratched',
    'cracked',
    'damaged',
    'water damage',
    'mold',
    'plays fine',    # Often signals undisclosed issues

    # Incomplete / non-standard items
    'no sleeve',
    'no cover',
    'coverless',
    'sleeve only',
    'cover only',
    'parts',
    'for parts',

    # Unofficial / low-value pressings
    'bootleg',
    'unofficial',
    'promo copy',
    'white label promo',
    'cut-out',
    'cutout',

    # CD / cassette / non-vinyl formats accidentally in results
    ' cd ',
    'cassette',
    '8-track',
    '8 track',

    # Vague listing quality signals
    'read description',
    'as-is',
    'as is',
    'untested'
]


In [ ]:
# Create a boolean mask to filter out listings with forbidden terms in the title
mask = ~items_df['Title'].str.lower().str.contains(
    r'(?:' + '|'.join(map(lambda t: t.strip(), forbidden_terms)) + r')',
    regex=True
)

# Apply the mask and reset the index
filtered_df = items_df[mask].reset_index(drop=True)
print(f'Listings after filtering: {len(filtered_df)}')
filtered_df


In [ ]:
# Save the filtered DataFrame to a CSV file named after the search term
safe_filename = search_term.replace(' ', '_').replace('/', '-') + '.csv'
filtered_df.to_csv(safe_filename, index=False)
print(f'Saved {len(filtered_df)} listings to: {safe_filename}')
